# 🥬 Training CNN 2D — Deteksi Kesiapan Panen Pakcoy
Jalankan semua sel dari atas ke bawah (Runtime > Run all). Di akhir kamu
akan dapat file **pakcoy_cnn2d.h5** yang otomatis ter-download.

Sebelum mulai: pastikan Runtime pakai GPU (Runtime > Change runtime type >
GPU) supaya lebih cepat — meski dataset kecil ini tetap cepat walau pakai CPU.

## 1. Upload dataset foto
Unggah `dataset_pakcoy_foto.zip` (dari Claude) saat diminta.

In [ ]:
from google.colab import files
import zipfile, os

print("Unggah dataset_pakcoy_foto.zip ...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

print("\nIsi folder dataset/:")
for d in sorted(os.listdir('dataset')):
    p = os.path.join('dataset', d)
    if os.path.isdir(p):
        print(f"  {d}: {len(os.listdir(p))} foto")


## 2. (Opsional) Tambah kelas "Siap Panen"
Kalau kamu punya foto untuk kelas ketiga, unggah zip foto-foto itu di sini
dan taruh di `dataset/siap_panen/`. Kalau tidak punya, lewati sel ini —
model akan tetap dilatih dengan 2 kelas yang ada.

In [ ]:
# LEWATI sel ini kalau belum punya foto "Siap Panen".
# Kalau punya, jalankan sel ini untuk upload zip foto siap_panen lalu
# extract ke dataset/siap_panen/

# from google.colab import files
# import zipfile, os
# uploaded = files.upload()
# zip_name = list(uploaded.keys())[0]
# os.makedirs('dataset/siap_panen', exist_ok=True)
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall('dataset/siap_panen')
# print(f"siap_panen: {len(os.listdir('dataset/siap_panen'))} foto")


## 3. Training CNN 2D
Arsitektur & preprocessing 100% sama dengan yang dibaca `app.py` (input 150x150, rescale /255, softmax).

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

DATASET_DIR = "dataset"
IMG_SIZE = (150, 150)      # HARUS 150x150 -- sesuai app.py
BATCH_SIZE = 8              # kecil karena dataset foto masih sedikit
EPOCHS = 40
SEED = 42
OUTPUT_MODEL = "pakcoy_cnn2d.h5"
VAL_SPLIT = 0.2

tf.random.set_seed(SEED)
np.random.seed(SEED)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=VAL_SPLIT, subset="training",
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int")
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=VAL_SPLIT, subset="validation",
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int")

class_names = train_ds.class_names
n_classes = len(class_names)
print(f"Kelas terdeteksi ({n_classes}): {class_names}")
print("-> Salin/urutan ini otomatis tersimpan ke class_order.json untuk app.py\n")

all_labels = np.array([int(y.numpy()) for _, y in train_ds.unbatch()])
classes_arr = np.arange(n_classes)
weights = compute_class_weight("balanced", classes=classes_arr, y=all_labels)
class_weight = dict(zip(classes_arr.tolist(), weights.tolist()))
print("Class weights:", class_weight)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(200).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

model = models.Sequential([
    layers.Rescaling(1.0/255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(n_classes, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                     class_weight=class_weight, callbacks=callbacks, verbose=1)

loss, acc = model.evaluate(val_ds, verbose=0)
print(f"\nAkurasi validasi akhir: {acc:.4f}  |  Loss validasi: {loss:.4f}")


## 4. Simpan model & label, lalu download

In [ ]:
model.save(OUTPUT_MODEL)

labels_display = [c.replace("_", " ").title() for c in class_names]
with open("class_order.json", "w") as f:
    json.dump(labels_display, f, indent=2, ensure_ascii=False)

print("Urutan label (salin ini kalau perlu, tapi app.py akan baca otomatis):")
print(", ".join(labels_display))

from google.colab import files
files.download(OUTPUT_MODEL)
files.download("class_order.json")


## 5. Selesai
Taruh `pakcoy_cnn2d.h5` dan `class_order.json` yang ter-download tadi
satu folder dengan `app.py`. Jalankan `streamlit run app.py` — halaman
"📸 Deteksi Siap Panen (CNN 2D)" otomatis pakai CNN ini (menggantikan
model scikit-learn bawaan sebelumnya, prioritasnya lebih tinggi).